<a href="https://colab.research.google.com/github/yusraanwar33-source/Deep-learning/blob/main/Library_management_system_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import csv  # Excel-like CSV file helper module
import os   # File checker module to see if a file exists

# DATA FILE NAMES (permenantly saved all of three)
BOOKS_FILE = "books.csv"      # File where all book stock numbers are stored
ISSUED_FILE = "issued.csv"    # File to check which student has which book right now
RETURNS_FILE = "returns.csv"  # File that logs old return history data line by line

def load_data(): # this function Brings old data from files into the program when it starts.
    books = {}    # Create an empty dictionary to hold book records
    issued = []   # Create an empty list to hold student loan records

    # LOAD BOOKS DATA
    if os.path.exists(BOOKS_FILE):  # If books.csv file is already there
        try:
            with open(BOOKS_FILE, "r", newline="", encoding="utf-8") as f:  # Open it to read
                reader = csv.DictReader(f)  # Setup the reader to read columns
                for row in reader:  # Check the file row by row by using loop
                    books[row["Book ID"]] = {  # Use Book ID as the main lock key
                        "name": row["Book Name"],  # Save the book name in row
                        "author": row["Author"],  # Save the author name in row
                        "quantity": int(row["Quantity Available"])  # Save stock count/quantity available as a number in row
                    }
        except (IOError, ValueError, KeyError):  #Handles errors.If the file has a typo error
            books = {}  # Start with an empty new book if file already not exist

    # LOAD ISSUED RECORDS
    if os.path.exists(ISSUED_FILE):  # If issued.csv file is already there
        try:
            with open(ISSUED_FILE, "r", newline="", encoding="utf-8") as f:  # Open it to read
                reader = csv.DictReader(f)  # Setup the reader to read columns wise.
                for row in reader:  # Go through every row in the file(loop use)
                    issued.append({  # Add this record data into our active list
                        "student_name": row["Student Name"],  # Save student name
                        "book_id": row["Book ID"],  # Save/load Book ID in row
                        "book_name": row["Book Name"],  # Save/load book name in row
                        "issued_qty": int(row["Issued Quantity"])  # Save/load the count as a number in row of issued qty (that is coloum) in issue file
                    })
        except (IOError, KeyError, ValueError):  # If there is a file error
            issued = []  # Start with a clean  new empty list is file not already exist.

    return books, issued  # Send all  the loaded data back to the main program

def save_data(books, issued): #Make function Saves the current Books and Issued data to files automatically
    try:
        # WRITE BOOKS DATA
        with open(BOOKS_FILE, "w", newline="", encoding="utf-8") as f:  # Open file in overwrite ('w') mode
            writer = csv.writer(f)  # Create a writer to type data in file
            writer.writerow(["Book ID", "Book Name", "Author", "Quantity Available"])  # Put column headers
            for b_id, details in books.items():  # Loop through all books one by one from dictonary
                writer.writerow([b_id, details["name"], details["author"], details["quantity"]])  # Save the book line,write row

        # WRITE ISSUED RECORDS
        with open(ISSUED_FILE, "w", newline="", encoding="utf-8") as f:  # Open file in overwrite mode
            writer = csv.writer(f)  # Create a writer to type_data/write in file
            writer.writerow(["Student Name", "Book ID", "Book Name", "Issued Quantity"])  # Put column headers /wrtie coloum names
            for record in issued:  # Loop through all student records one by one from list
                writer.writerow([record["student_name"], record["book_id"], record["book_name"], record["issued_qty"]])  # Save student line/row write

        print("[Auto-Save] Data successfully saved to CSV files!")  # Show a success message
    except IOError:  # If the system blocks saving the file
        print("[System Error] Auto-save failed!")  # Show an eror message

def log_return_to_csv(student_name, book_id, book_name, total_issued, ret_qty, remaining):
    #Adds a new return history line at the bottom of returns.csv file.OR"Append/add all returned books data into returns.csv"
    try:
        file_exists = os.path.exists(RETURNS_FILE)  # Check if returns.csv file is already there

        with open(RETURNS_FILE, "a", newline="", encoding="utf-8") as f:  # Open in append ('a') mode to not delete old lines but add new data
            writer = csv.writer(f)  # Setup/create file writer tool
            if not file_exists:  # If the file is brand new create headings
                writer.writerow(["Student Name", "Book ID", "Book Name", "Total Issued", "Returned Now", "Remaining Still"])  # Make columns

            writer.writerow([student_name, book_id, book_name, total_issued, ret_qty, remaining])  # Write the details line in row and save it

        print("[History] Return record successfully added to returns.csv!")  # Show success message
    except IOError:  # If file cannot be written
        print("[System Error] Could not write to returns history file.")  # Show error message

def add_book(books, issued):
    # Option 1: Add a brand new book or change an old book entry
    print("\n--- Add / Update Book ---")
    book_id = input("Enter Book ID: ").strip()  # Ask for Book ID from user and remove extra spaces

    if book_id in books:  #checking If this Book ID is already in the library stock
        print(f"\n[System] This Book ID ({book_id}) already exists!")  # Warning alert
        print(f"Current Details -> Name: {books[book_id]['name']} | Author: {books[book_id]['author']} | Qty: {books[book_id]['quantity']}")  # Show old data
        print("\nWhat do you want to do?")
        print("1. Just add more quantity")
        print("2. Correct/Change Book Name and Author (Edit Entry)")
        print("3. Cancel and go back")

        sub_choice = input("Enter option (1-3): ").strip()  # Get user choice numbers

        if sub_choice == '1':  # If user just wants to add more stock
            try:
                qty = int(input("Enter additional quantity to add: "))  # Ask how many more books
                if qty < 0: raise ValueError  # Block minus numbers
                books[book_id]["quantity"] += qty  # Add new books count to the old stock count
                print(f"Quantity updated! Total now: {books[book_id]['quantity']}")  # Show new total stock
                save_data(books, issued)  # Save changes to file automatically
            except ValueError:  # If input is not a clear number
                print("[Error] Invalid quantity. Cancelled.")  # Show error message

        elif sub_choice == '2':  # If user wants to edit wrong spelling name or author
            new_name = input("Enter CORRECT Book Name: ").strip()  # Get correct name
            new_author = input("Enter CORRECT Author Name: ").strip()  # Get correct author
            try:
                new_qty = int(input("Enter Correct Quantity: "))  # Get correct total quantity
                if new_qty < 0: raise ValueError  # Block minus numbers
                books[book_id] = {  # Put new data over the old wrong entry data
                    "name": new_name,
                    "author": new_author,
                    "quantity": new_qty
                }
                print(f"[Success] Book ID {book_id} has been corrected!")  # Success message
                save_data(books, issued)  # Save changes to file automatically
            except ValueError:  # If input count is wrong
                print("[Error] Invalid quantity. Correction failed.")  # Show error message
        else:
            print("Operation cancelled.")  # Cancel message

    else:  # If it is a completely new Book ID
        name = input("Enter Book Name: ").strip()  # Get book name
        author = input("Enter Author: ").strip()  # Get author name
        try:
            qty = int(input("Enter Quantity Available: "))  # Get stock number
            if qty < 0: raise ValueError  # Block minus numbers
            books[book_id] = {  # Save the new profile details in the system data
                "name": name,
                "author": author,
                "quantity": qty
            }
            print(f"Book '{name}' added successfully!")  # Success message
            save_data(books, issued)  # Save the fresh file to database
        except ValueError:  # If quantity is wrong
            print("[Error] Invalid quantity. Book not added.")  # Show error message

def issue_book(books, issued):
    #Option 2: Lend books to students with check limits
    print("\n--- Issue a Book ---")
    student_name = input("Enter Student Name: ").strip()  # Get student name
    book_id = input("Enter Book ID to issue: ").strip()  # Get requested book ID

    if book_id not in books:  # Check 1: If book does not exist in library
        print("[Error] Book ID not found in library.")  # Show error
        return  # Stop function here

    if books[book_id]["quantity"] <= 0:  # Check 2: If stock count is zero
        print("[Error] Sorry, this book is currently out of stock.")  # Show error
        return  # Stop function here

    try:
        req_qty = int(input(f"How many copies do you want? (Available in stock: {books[book_id]['quantity']}): "))  # Get requested count number

        if req_qty <= 0:  # Check 3: If user types 0 or minus number
            print("[Error] Quantity must be greater than 0.")  # Show error
            return  # Stop function here

        if req_qty > books[book_id]["quantity"]:  # Check 4: If student asks for more books than stock pool
            print(f"[Error] Not enough stock! We only have {books[book_id]['quantity']} copies available.")  # Show warning
            return  # Stop function here

        books[book_id]["quantity"] -= req_qty  # Minus the given count from library shelf stock count Or reduce quantity of booksform libaray stock

        record = {  # Make an active loan tracker data packet OR create new record of students
            "student_name": student_name,
            "book_id": book_id,
            "book_name": books[book_id]["name"],
            "issued_qty": req_qty
        }
        issued.append(record)  # Add this packet into our active issued list
        print(f"[Success] {req_qty} copies of '{books[book_id]['name']}' issued to {student_name}!")  # Success message
        save_data(books, issued)  # Save background data files automaicaly

    except ValueError:  # If text or ABC is typed instead of numbers
        print("[Error] Please enter a valid number for quantity.")  # Show error message

def return_book(books, issued):
    #Option 3: Get books back from student, find math counts and save to returns.csv
    print("\n--- Return a Book ---")
    student_name = input("Enter Student Name: ").strip()  # Get student name
    book_id = input("Enter Book ID to return: ").strip()  # Get book ID

    found_record = None  # Create a clean variable to hold any matching record found
    for record in issued:  # Loop search inside the active issued list
        if record["student_name"].lower() == student_name.lower() and record["book_id"] == book_id:  # If matching student name and book ID is found
            found_record = record  # Store the matching entry details
            break  # Stop search loop instantly

    if found_record:  # If a matching record is actually found
        total_originally_issued = found_record['issued_qty']  # Check how many books they borrowed at first
        print(f"\n[System] Found Record: This student has issued {total_originally_issued} copies of this book.")  # Print details
        try:
            ret_qty = int(input("How many copies are you returning now?: "))  # Ask how many copies they are giving back now

            if ret_qty <= 0:  # Check 1: If return number is 0 or minus
                print("[Error] Return quantity must be greater than 0.")  # Show error
                return  # Stop function here

            if ret_qty > total_originally_issued:  # Check 2: If student tries to return more than they borrowed
                print(f"[Error] Invalid quantity! Student only has {total_originally_issued} copies.")  # Show error
                return  # Stop function here

            if book_id in books:  # If book ID matches active library stock record
                books[book_id]["quantity"] += ret_qty  # Add returned copies back into the library stock count

            remaining_still_with_student = total_originally_issued - ret_qty  # Math calculate how many books are still left with them
            found_record["issued_qty"] -= ret_qty  # Subtract the returned books from active loan register list

            # ##ADVANCED STEP: SEND DATA LOG TO HISTORIES FILE ###
            # Send all math data numbers to log function for permanent background save
            log_return_to_csv(
                student_name=record["student_name"],
                book_id=book_id,
                book_name=found_record["book_name"],
                total_issued=total_originally_issued,
                ret_qty=ret_qty,
                remaining=remaining_still_with_student
            )

            if found_record["issued_qty"] == 0:  # If student has zero books left with them now
                issued.remove(found_record)  # Remove student completely from active loan list register
                print(f"[Success] All copies successfully returned by {student_name}!")  # Show full check-in success message
            else:  # If student still keeps some books at home
                print(f"[Success] {ret_qty} copy returned. Remaining {found_record['issued_qty']} copies are STILL issued to {student_name}.")  # Show bachi hui quantity status

            save_data(books, issued)  # Save Automatic files update

        except ValueError:  # If string or ABC text is typed in input fields
            print("[Error] Please enter a valid number.")  # Show error message
    else:  # If student name or book ID match fails completely
        print("[Error] No matching active issue record found for this student.")  # Show error message

def search_book(books):
    # Option 4: Find books in storage using Name or Author words
    print("\n--- Search Book ---")
    print("1. Search by Book Name")
    print("2. Search by Author")
    choice = input("Choose search criteria (1-2): ").strip()  # Get choice method number

    if choice not in ['1', '2']:  # If wrong option number is entered
        print("[Error] Invalid choice.")  # Show error
        return  # Stop function here

    query = input("Enter your search keyword: ").strip().lower()  # Get search keyword in simple small letters
    found = False  # Set variable to track if match is found or not

    print("\nSearch Results:")
    print(f"{'ID':<10} {'Book Name':<30} {'Author':<25} {'Qty':<5}")  # Design table format headers with spaces
    print("-" * 72)  # Layout line breaker

    for b_id, details in books.items():  # Check each book inside library data dict
        match = False  # Set default single row match tracker as false
        if choice == '1' and query in details["name"].lower():  # If keyword matches the book title name string
            match = True  # Set row hit as true
        elif choice == '2' and query in details["author"].lower():  # If keyword matches the author name string
            match = True  # Set row hit as true

        if match:  # If book row matched successfully
            print(f"{b_id:<10} {details['name']:<30} {details['author']:<25} {details['quantity']:<5}")  # Print aligned table row
            found = True  # Mark global search find as true

    if not found:  # If no match is found after scanning whole list data
        print("No matching books found.")  # Show empty notification message

def display_reports(books, issued):
    #Option 5: Calculate total items count metrics on screen or shows libabray total report on screen
    print("\n=====LIBRARY REPORT=====")
    total_distinct_books = len(books)  # Count absolute unique titles stored
    total_physical_books = sum(item["quantity"] for item in books.values())  # Add total physical books on library shelves right now
    total_issued = sum(record["issued_qty"] for record in issued)  # Add total outstanding copies out with students on loan right now

    print(f"Total Unique Titles Stored: {total_distinct_books}")  # Print count results
    print(f"Total Physical Books Currently Available: {total_physical_books}")  # Print count results
    print(f"Total Books Currently Issued Out      : {total_issued}")  # Print count results
    print("================================")

def main():
    ####Main program execution loop control center menu OR main control room from where all program run####
    books, issued = load_data()  # Load all base_background?permenant files at start of program execution

    while True:  # Run loop non-stop until Option 6 is selected by user
        print("\n===== LIBRARY MANAGEMENT SYSTEM =====")
        print("1. Add/Update Books")
        print("2. Issue Book")
        print("3. Return Book")
        print("4. Search Book")
        print("5. Display Reports")
        print("6. Exit Program")
        print("=====================================")

        choice = input("Enter your choice (1-6): ").strip()  # Take selected menu options navigation path

        if choice == '1':
            add_book(books, issued)  # Trigger add feature lines
        elif choice == '2':
            issue_book(books, issued)  # Trigger issue feature lines
        elif choice == '3':
            return_book(books, issued)  # Trigger return processing logic lines
        elif choice == '4':
            search_book(books)  # Trigger word filter search lines
        elif choice == '5':
            display_reports(books, issued)  # Trigger total math data logs screen printer
        elif choice == '6':
            print("\nExiting program. All data is already saved. Goodbye!")  # Goodbye screen notice
            break  # Break runtime loop to stop program safely
        else:
            print("[Error] Invalid choice! Please select between 1 and 6.")  # Input safety catch message

# Run main controller automatically if script is executed directly
if __name__ == "__main__":
    main()


===== LIBRARY MANAGEMENT SYSTEM =====
1. Add/Update Books
2. Issue Book
3. Return Book
4. Search Book
5. Display Reports
6. Exit Program
Enter your choice (1-6): 1

--- Add / Update Book ---
Enter Book ID: #04
Enter Book Name: Black magic
Enter Author: Anonymous
Enter Quantity Available: 1
Book 'Black magic' added successfully!
[Auto-Save] Data successfully saved to CSV files!

===== LIBRARY MANAGEMENT SYSTEM =====
1. Add/Update Books
2. Issue Book
3. Return Book
4. Search Book
5. Display Reports
6. Exit Program
Enter your choice (1-6): Love nature
[Error] Invalid choice! Please select between 1 and 6.

===== LIBRARY MANAGEMENT SYSTEM =====
1. Add/Update Books
2. Issue Book
3. Return Book
4. Search Book
5. Display Reports
6. Exit Program
Enter your choice (1-6): 1

--- Add / Update Book ---
Enter Book ID: #05
Enter Book Name: Love nature
Enter Author: Bella
Enter Quantity Available: 100
Book 'Love nature' added successfully!
[Auto-Save] Data successfully saved to CSV files!

===== LIB